# Arrow Securities and State Prices

This notebook develops the pricing of state-contingent claims from expected utility. The central idea is simple:

> A unit of consumption is especially valuable in states where marginal utility is high.

We begin with a one-period asset-pricing equation, connect it to von Neumann--Morgenstern expected utility, and then price Arrow securities in a finite-state Markov economy.

## 1. Pricing a one-period asset

Let current consumption be $c_0$. Next-period consumption is uncertain and can take the state-contingent values

$$
c_1,\ldots,c_n,
$$

with probabilities $\pi_1,\ldots,\pi_n$. An asset costs $q$ units of current consumption and pays $r_i$ units of consumption next period if state $i$ occurs.

If the household purchases $z$ units of the asset, expected discounted utility is

$$
u(c_0-qz)+\beta\sum_{i=1}^n \pi_i u(c_i+r_i z).
$$

The first-order condition is

$$
-q u'(c_0-qz)+\beta\sum_{i=1}^n\pi_i u'(c_i+r_i z)r_i=0.
$$

For a marginal trade evaluated at $z=0$,

$$
q=\beta\sum_{i=1}^n\pi_i\frac{u'(c_i)}{u'(c_0)}r_i.
$$

Define the stochastic discount factor

$$
m_i=\beta\frac{u'(c_i)}{u'(c_0)}.
$$

Then

$$
\boxed{q=\sum_{i=1}^n\pi_i m_i r_i=E[m r].}
$$

Thus an asset is valuable not only when its payoff is large, but especially when it pays in states where marginal utility is high.

Reference: John H. Cochrane, *Asset Pricing*, revised edition, Princeton University Press, 2009.

## 2. Expected utility and lotteries

The von Neumann--Morgenstern expected-utility theorem provides a representation of preferences over lotteries. Under the standard axioms, a lottery that delivers outcome $A_i$ with probability $p_i$ can be evaluated as

$$
E[u(A)]=\sum_i p_i u(A_i).
$$

For macroeconomics, the outcomes can be state-contingent future consumption. If the possible states at date $t$ are

$$
S=\{s_1,\ldots,s_n\},
$$

then consumption is a function of the realized state,

$$
c_t=c_t(s_i).
$$

If state $s_i$ occurs with conditional probability $\pi_i$, expected discounted felicity at date $t$ is

$$
E_0[\beta^t u(c_t)]
=\beta^t\sum_{i=1}^n\pi_i u(c_t(s_i)).
$$

This is the bridge from ordinary intertemporal choice to asset pricing under uncertainty.

## 3. CRRA felicity and marginal utility

We use CRRA felicity,

$$
u(c)=
\begin{cases}
\dfrac{c^{1-\gamma}-1}{1-\gamma}, & \gamma\neq1,\\[6pt]
\log c, & \gamma=1,
\end{cases}
$$

where $\gamma>0$ is relative risk aversion and $1/\gamma$ is the intertemporal elasticity of substitution in the standard time-separable CRRA model.

Marginal utility is particularly simple:

$$
\boxed{u'(c)=c^{-\gamma}.}
$$

In [1]:
import numpy as np


def utility(c, gamma):
    """CRRA felicity."""
    c = np.asarray(c, dtype=float)
    if np.any(c <= 0):
        raise ValueError("Consumption must be strictly positive.")
    if np.isclose(gamma, 1.0):
        return np.log(c)
    return (c ** (1.0 - gamma) - 1.0) / (1.0 - gamma)


def marginal_utility(c, gamma):
    """Marginal utility under CRRA preferences."""
    c = np.asarray(c, dtype=float)
    if np.any(c <= 0):
        raise ValueError("Consumption must be strictly positive.")
    return c ** (-gamma)

## 4. Discounted utility of a deterministic consumption path

For a deterministic path $(c_0,\ldots,c_T)$,

$$
U(c_0,\ldots,c_T)=\sum_{t=0}^T\beta^t u(c_t).
$$

A vectorized implementation makes the formula transparent.

In [2]:
def lifetime_utility(consumption, beta, gamma):
    """Discounted utility of a deterministic consumption path."""
    consumption = np.asarray(consumption, dtype=float)
    discount_factors = beta ** np.arange(consumption.size)
    return np.sum(discount_factors * utility(consumption, gamma))

## 5. Arrow securities in a Markov economy

An Arrow security pays **one unit of consumption in one particular future state and zero otherwise**.

Suppose the current state is $i$ and the security pays one unit at date $t$ only if state $j$ occurs. Let $P$ be the one-period Markov transition matrix. Then

$$
\Pr(s_t=j\mid s_0=i)=(P^t)_{ij}.
$$

The date-0 price of this claim is

$$
q_{ij}^{(t)}
=(P^t)_{ij}\,\beta^t\frac{u'(c_j)}{u'(c_i)}.
$$

Under CRRA utility,

$$
\boxed{
q_{ij}^{(t)}=(P^t)_{ij}\,\beta^t
\left(\frac{c_j}{c_i}\right)^{-\gamma}.
}
$$

For $t=1$, these are the familiar one-period state prices.

**Important qualification.** The $t>1$ formula below assumes consumption is determined by the state alone, so the marginal-utility ratio between dates 0 and $t$ depends only on $c_i$ and $c_j$.

In [3]:
def validate_markov_matrix(P, atol=1e-12):
    """Check that P is a square row-stochastic matrix."""
    P = np.asarray(P, dtype=float)
    if P.ndim != 2 or P.shape[0] != P.shape[1]:
        raise ValueError("P must be a square matrix.")
    if np.any(P < -atol) or np.any(P > 1.0 + atol):
        raise ValueError("Transition probabilities must lie in [0, 1].")
    if not np.allclose(P.sum(axis=1), 1.0, atol=atol):
        raise ValueError("Each row of P must sum to one.")
    return P


def arrow_price(consumption, beta, gamma, P, horizon, current_state, payoff_state):
    """Price a t-period Arrow security in a finite-state Markov economy."""
    P = validate_markov_matrix(P)
    consumption = np.asarray(consumption, dtype=float)

    if consumption.ndim != 1 or consumption.size != P.shape[0]:
        raise ValueError("Consumption must have one positive value for each state.")
    if np.any(consumption <= 0):
        raise ValueError("Consumption must be strictly positive.")
    if not (0 < beta <= 1):
        raise ValueError("beta must lie in (0, 1].")
    if horizon < 0 or int(horizon) != horizon:
        raise ValueError("horizon must be a nonnegative integer.")

    horizon = int(horizon)
    P_t = np.linalg.matrix_power(P, horizon)
    probability = P_t[current_state, payoff_state]
    sdf = beta**horizon * (
        marginal_utility(consumption[payoff_state], gamma)
        / marginal_utility(consumption[current_state], gamma)
    )
    return probability * sdf

## 6. Numerical example

Consider four consumption states,

$$
c=(0.5,1,1,1),
$$

with $\beta=0.97$ and $\gamma=2$. We price a one-period Arrow security when the current state is $i=1$ and the security pays one unit if next period's state is $j=0$.

In [4]:
consumption = np.array([0.5, 1.0, 1.0, 1.0])
beta = 0.97
gamma = 2.0

P = np.array([
    [0.25, 0.00, 0.25, 0.50],
    [0.15, 0.10, 0.25, 0.50],
    [0.25, 0.30, 0.00, 0.45],
    [0.25, 0.25, 0.25, 0.25],
])

horizon = 1
current_state = 1
payoff_state = 0

price = arrow_price(
    consumption, beta, gamma, P,
    horizon, current_state, payoff_state
)

print(f"Arrow security: pays 1 unit in state {payoff_state} at t={horizon}")
print(f"Current state: {current_state}")
print(f"Transition probability: {P[current_state, payoff_state]:.3f}")
print(f"Arrow price: {price:.4f}")

Arrow security: pays 1 unit in state 0 at t=1
Current state: 1
Transition probability: 0.150
Arrow price: 0.5820


### Why is the price relatively high?

From current state $1$, consumption is $c_i=1$. The Arrow security pays in state $0$, where consumption is only $c_j=0.5$. With $\gamma=2$,

$$
\frac{u'(0.5)}{u'(1)}
=\left(\frac{0.5}{1}\right)^{-2}=4.
$$

The claim therefore pays precisely in a state where marginal utility is four times as high. Its price is

$$
q_{10}=P_{10}\beta\frac{u'(c_0)}{u'(c_1)}
=0.15(0.97)(4)=0.582.
$$

This illustrates the central asset-pricing principle: **payoffs in bad consumption states command high prices.**

## 7. The complete one-period state-price matrix

Rather than price one Arrow security at a time, we can construct every one-period state price at once. Define

$$
Q_{ij}=P_{ij}\beta\frac{u'(c_j)}{u'(c_i)}.
$$

The matrix $Q$ is the **state-price matrix**. Entry $Q_{ij}$ is the current price, in state $i$, of one unit of consumption delivered next period only in state $j$.

In [5]:
def state_price_matrix(consumption, beta, gamma, P):
    """Construct the one-period Arrow state-price matrix Q."""
    P = validate_markov_matrix(P)
    consumption = np.asarray(consumption, dtype=float)
    mu = marginal_utility(consumption, gamma)
    sdf = beta * mu[None, :] / mu[:, None]
    return P * sdf


Q = state_price_matrix(consumption, beta, gamma, P)
np.set_printoptions(precision=4, suppress=True)
print("State-price matrix Q:")
print(Q)

State-price matrix Q:
[[0.2425 0.     0.0606 0.1212]
 [0.582  0.097  0.2425 0.485 ]
 [0.97   0.291  0.     0.4365]
 [0.97   0.2425 0.2425 0.2425]]


## 8. Connection to the risk-free asset

A one-period risk-free bond pays one unit of consumption in **every** state next period. Therefore its price in current state $i$ is the sum of the Arrow prices in row $i$:

$$
q_i^f=\sum_j Q_{ij}.
$$

Its gross risk-free return is

$$
\boxed{R_i^f=\frac{1}{q_i^f}.}
$$

Thus even though the bond's payoff is risk free, its price and return can depend on the current state.

In [6]:
bond_prices = Q.sum(axis=1)
risk_free_returns = 1.0 / bond_prices

for state, (q_f, R_f) in enumerate(zip(bond_prices, risk_free_returns)):
    print(
        f"State {state}: bond price = {q_f:.4f}, "
        f"gross risk-free return = {R_f:.4f}, "
        f"net return = {R_f - 1:.2%}"
    )

State 0: bond price = 0.4244, gross risk-free return = 2.3564, net return = 135.64%
State 1: bond price = 1.4065, gross risk-free return = 0.7110, net return = -28.90%
State 2: bond price = 1.6975, gross risk-free return = 0.5891, net return = -41.09%
State 3: bond price = 1.6975, gross risk-free return = 0.5891, net return = -41.09%


## 9. Takeaway

The sequence of ideas is

$$
\boxed{
\text{expected utility}
\longrightarrow
\text{marginal utility}
\longrightarrow
\text{stochastic discount factor}
\longrightarrow
\text{Arrow prices}
\longrightarrow
\text{state-price matrix}.
}
$$

For a one-period payoff vector $x$, the price in each current state is simply

$$
\boxed{v=Qx.}
$$

This matrix formulation is the natural starting point for recursive asset pricing in a finite-state Markov economy.